# Te enseño a scrapear páginas de fútbol
Notebook de las demos del video de YouTube. **Antes de grabar:** correr `python chequeo_previo.py` en esta carpeta.
Si una página falla en cámara, cada demo usa sola el respaldo guardado (y lo avisa).

In [1]:
import requests, json, os, sys, asyncio, threading
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36',
    'Accept-Language': 'es-AR,es;q=0.9,en;q=0.8',
}
RESPALDO = 'respaldo'

def respaldo(nombre):
    ruta = os.path.join(RESPALDO, nombre)
    print(f'(usando respaldo: {ruta})')
    if ruta.endswith('.json'):
        return json.load(open(ruta, encoding='utf-8'))
    if ruta.endswith('.csv'):
        return pd.read_csv(ruta)
    return open(ruta, encoding='utf-8').read()

## 1. El árbol de decisión (para cualquier página nueva)
1. ¿Está en el HTML? → `requests` + `BeautifulSoup`
2. ¿Hay una API en Network? → `requests` directo a esa URL, leyendo JSON
3. ¿Ninguna, o me bloquean? → ahí entra el navegador automatizado

Cómo saberlo: **Ctrl+U** (código fuente: ¿está el dato?) y **F12 → Network → Fetch/XHR** (¿hay un request que trae el JSON?).

## 2. Caso HTML: Transfermarkt
Ctrl+U → buscar "River Plate": el dato ya viene en el HTML. Hay que pedir la página y "leerla".

In [9]:
url_tm = 'https://www.transfermarkt.com/torneo-clausura/startseite/wettbewerb/ARGC'
r_tm = requests.get(url_tm, headers=HEADERS, timeout=30)
print(r_tm.status_code)
r_tm.text[:300]          # lo que llega: HTML, texto con etiquetas

200


'<!DOCTYPE html>\n<html lang="en">\n\n<head>\n    \n<script type="text/javascript" data-description="sourcepoint stub code">\n    !function () { var e = function () { var e, t = "__tcfapiLocator", a = [], n = window; for (; n;) { try { if (n.frames[t]) { e = n; break } } catch (e) { } if (n === window.top)'

In [3]:
try:
    soup = BeautifulSoup(r_tm.text, 'html.parser')
    tabla = soup.find('table', class_='items')            # buscar la tabla en el HTML
    clubes = []
    for fila in tabla.find('tbody').find_all('tr'):
        link = fila.select_one('td.hauptlink a')
        if link is None:
            continue
        celdas = fila.find_all('td')
        clubes.append({'club': link.get_text(strip=True),
                       'plantel': celdas[2].get_text(strip=True),
                       'valor_total': celdas[-1].get_text(strip=True)})
    clubes = pd.DataFrame(clubes)
except Exception as e:
    print('Transfermarkt falló:', e)
    clubes = respaldo('transfermarkt.csv')
clubes.head(10)

,club,plantel,valor_total
0,CA River Plate,26,€124.50M
1,CA Boca Juniors,32,€111.53M
2,CA Rosario Central,30,€63.00M
3,Racing Club,27,€62.00M
4,Club Estudiantes de La Plata,31,€54.78M
5,CS Independiente Rivadavia,29,€43.13M
6,CA Vélez Sarsfield,25,€39.70M
7,Club Atlético Tigre,30,€38.28M
8,CA San Lorenzo de Almagro,33,€38.25M
9,CA Talleres,29,€36.92M


## 3. Caso API: 365Scores
F12 → Network → Fetch/XHR → refrescar → el request `stats` trae todo en JSON. No hay que "leer" ninguna página: pedís esa URL y listo.

In [4]:
url_365 = ('https://webws.365scores.com/web/stats/?appTypeId=5&langId=29&timezoneName=America/Buenos_Aires'
           '&userCountryId=382&competitions=72&competitors=&withSeasons=true')
try:
    datos = requests.get(url_365, headers=HEADERS, timeout=30).json()   # lo que llega: JSON, ya ordenado
except Exception as e:
    print('365Scores falló:', e)
    datos = respaldo('365scores.json')
rankings = datos['stats']['athletesStats']['tables']
[t['name'] for t in rankings]

['Goles',
 'Goles esperados',
 'Asistencias',
 'Asistencias esperadas',
 'Goles y Asistencias',
 'Goles esperados y asistencias',
 'Puntajes 365',
 'Penales convertidos',
 'Barridas ganadas',
 'Intercepciones',
 'Tarjetas Rojas',
 'Tarjetas Amarillas',
 'Valla invicta',
 'Goles recibidos',
 'Salvadas',
 'Penales atajados']

In [5]:
goles = rankings[0]
pd.DataFrame([{'jugador': f['entity']['name'], 'goles': f['stats'][0]['value']} for f in goles['rows']]).head(10)

,jugador,goles
0,Ronaldo Martínez,10
1,Agustin Modica,10
2,Cristian Tarragona,9
3,Junior Marabel,8
4,Andres Vombergar,7
5,Álex Arce,7
6,Jose Florentín,6
7,Agustín Auzmendi,6
8,Jordy Caicedo,6
9,Ángel Di María,6


**HTML vs API, la diferencia:**

| | HTML (Transfermarkt) | API (365Scores) |
|---|---|---|
| Qué llega | La página entera, con diseño | Solo los datos, en JSON |
| Cómo se encuentra | Ctrl+U | F12 → Network → Fetch/XHR |
| Qué hay que hacer | Buscar la tabla con BeautifulSoup | Recorrer el diccionario |
| Qué se rompe | Si cambian el diseño | Si cambian la API |

Si hay API, casi siempre conviene la API.

## 4. Cuando me bloquean: el navegador automatizado
Python maneja un Chrome de verdad: abre la página como lo harías vos y te da el HTML.

| Herramienta | En una línea |
|---|---|
| **Selenium** | La clásica. Muchísimos tutoriales, pero las páginas la detectan fácil |
| **Playwright** | La moderna (de Microsoft): espera sola a que cargue todo, más rápida |
| **undetected-chromedriver** | Selenium "disfrazado" para que no se note que es un robot |
| **nodriver** / **pydoll** | Lo más nuevo: hablan directo con Chrome, sin las marcas de automatización |

Ejemplo con **FBref**: con `requests` no te deja entrar.

In [6]:
url_fbref = 'https://fbref.com/en/comps/21/Primera-Division-Stats'
print('FBref con requests →', requests.get(url_fbref, headers=HEADERS, timeout=30).status_code)

FBref con requests → 403


In [14]:
# Con un navegador automatizado (pydoll): abre Chrome, entra a la página y me da el HTML
from pydoll.browser import Chrome

async def abrir_con_navegador(url):
    async with Chrome() as browser:
        tab = await browser.start()
        await tab.go_to(url)
        await asyncio.sleep(5)                       # que termine de cargar
        r = await tab.execute_script('return document.documentElement.outerHTML;')
        return r['result']['result']['value'] if isinstance(r, dict) else r

def correr(coro, tope=90):
    # Jupyter en Windows no puede abrir Chrome desde su propio loop: lo corremos en un hilo aparte
    loop = asyncio.ProactorEventLoop() if sys.platform == 'win32' else asyncio.new_event_loop()
    threading.Thread(target=loop.run_forever, daemon=True).start()
    return asyncio.run_coroutine_threadsafe(coro, loop).result(timeout=tope)

try:
    html_fbref = correr(abrir_con_navegador(url_fbref))
    assert '<table' in html_fbref, 'llegó sin tablas'
except Exception as e:
    print('El navegador falló:', type(e).__name__, e)
    html_fbref = respaldo('fbref.html')
print('Tablas en la página:', len(BeautifulSoup(html_fbref, 'html.parser').find_all('table')))

Tablas en la página: 27


In [8]:
# Y de ahí en adelante es igual que el caso HTML
soup_fb = BeautifulSoup(html_fbref, 'html.parser')
tabla = soup_fb.find('table', id='stats_squads_standard_for')
equipos = pd.read_html(StringIO(str(tabla)))[0]
equipos.columns = [b if a.startswith('Unnamed') else f'{a} {b}' for a, b in equipos.columns]
equipos[['Squad', 'Poss', 'Performance Gls', 'Performance Ast', 'Performance CrdY']].sort_values('Performance Gls', ascending=False).head(10)

,Squad,Poss,Performance Gls,Performance Ast,Performance CrdY
28,Unión,48.6,47,34,76
16,Ind. Rivadavia,47.7,47,32,74
6,Boca Juniors,60.7,46,36,72
24,Rosario Central,57.5,43,34,73
13,Gimnasia–LP,46.1,43,31,85
29,Vélez Sarsfield,53.7,42,30,70
23,River Plate,64.3,42,26,71
18,Instituto,47.9,41,26,86
9,Defensa,51.9,39,28,63
17,Independiente,51.6,38,27,77


## 5. ¿Por qué me bloquean? Cloudflare (solo charla, sin código)
Muchas páginas (FBref, FotMob, Sofascore…) tienen un "patovica" delante: Cloudflare o similares. Mira tres cosas:
1. **Cómo saludás**: la huella de tu conexión. La de `requests` no se parece a la de ningún navegador.
2. **Si sos un navegador de verdad**: te hace resolver un desafío de JavaScript ("Verificando que sos humano…").
3. **Cuánto pedís**: si pedís muy rápido, te banea un rato (FBref: ~10 páginas por minuto).

Por eso el paso 3 del árbol es el navegador automatizado: pasa por un navegador de verdad. Y por eso, aunque entres, **pedí despacio**.

## 6. Resumen
1. ¿Está en el HTML? → `requests` + `BeautifulSoup` (Transfermarkt)
2. ¿Hay API en Network? → `requests` + JSON (365Scores)
3. ¿Me bloquean? → navegador automatizado: Selenium, Playwright, nodriver, pydoll (FBref)
